# Best place to be

## 1. Introduction
### Contexte du projet
Uber, acteur mondial du transport à la demande, cherche à réduire les temps d’attente des clients en anticipant la demande et en recommandant aux chauffeurs les zones où se concentrer à chaque instant de la journée.

Ce projet s’inscrit dans cette démarche : il s’agit de détecter les zones chaudes (hotspots) de prise en charge à New York, en utilisant des méthodes d’apprentissage non supervisé.

### Objectifs

- Identifier les zones à forte demande à partir des données de trajets Uber à New York.
- Comparer plusieurs algorithmes de clustering pour trouver les zones les plus pertinentes.
- Visualiser les zones chaudes sur une carte interactive (Plotly / Mapbox).
- Fournir une base analytique permettant à Uber d’optimiser la répartition de ses chauffeurs selon l’heure et le jour.

### Données

- Jeu de données : Uber pickups à New York (2015)
- Variables principales :
  - Lat, Lon : coordonnées GPS de la prise en charge
  - Datetime : date et heure du trajet
  - Base : base dispatchant les chauffeurs

Les données couvrent plusieurs jours et périodes de la journée, permettant une analyse spatio-temporelle de la demande.

### Bibliothèques

In [25]:
import pandas as pd
import io # pour gérer les flux de données en mémoire
import zipfile
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import plotly.express as px
import hdbscan

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)



### Chargement

In [26]:
# Chargement du fichier zip principal
outer_zip_path = "D:/Profils/NLefort/Desktop/JEDHA/PROJETS/06.ML_non_supervise/uber-trip-data.zip"
sample_size = 10000 # récupère un échantillon de 10 000 lignes par fichier
sample_df = pd.DataFrame() # pour stocker les échantillons

In [27]:
# Nom du zip interne
inner_zip_name = "uber-trip-data/uber-raw-data-janjune-15.csv.zip"

# Lecture du fichier zip imbriqué
with zipfile.ZipFile(outer_zip_path) as outer_zip:
    with outer_zip.open(inner_zip_name) as inner_zip_file:
        inner_zip_bytes = inner_zip_file.read()
        with zipfile.ZipFile(io.BytesIO(inner_zip_bytes)) as inner_zip:
            
            csv_name = inner_zip.namelist()[0]  # le CSV interne
            df = pd.read_csv(inner_zip.open(csv_name), encoding="latin1")
            
            n_sample = min(sample_size, len(df))
            sample_df_2015 = df.sample(n=n_sample, random_state=42).reset_index(drop=True)

print(sample_df_2015.head())
print(f"Échantillon 2015 contient {len(sample_df_2015)} lignes")


  Dispatching_base_num          Pickup_date Affiliated_base_num  locationID
0               B02764  2015-04-04 23:39:00              B02764          41
1               B02617  2015-03-23 08:35:00              B02617         164
2               B02764  2015-06-20 20:01:00              B02764         163
3               B02764  2015-02-09 19:55:29              B02725         129
4               B02617  2015-06-09 21:25:00              B02617         132
Échantillon 2015 contient 10000 lignes


In [28]:
# Chargement des autres fichiers zip
sample_df_2014 = pd.DataFrame()  # <-- vide pour ne contenir que 2014

with zipfile.ZipFile(outer_zip_path) as z:
    for file_name in z.namelist():
        if "taxi-zone-lookup" in file_name.lower():
            continue
        if file_name.endswith('.csv') and 'uber-raw-data' in file_name.lower():
            if "15" in file_name:  # <-- ignore explicitement les fichiers 2015
                continue

            df = pd.read_csv(z.open(file_name), encoding='latin1')
            
            n_sample = min(sample_size, len(df))
            sample_df_2014 = pd.concat(
                [sample_df_2014, df.sample(n=n_sample, random_state=42)],
                ignore_index=True
            )

print(sample_df_2014.head())
print(f"Échantillon 2014 contient {len(sample_df_2014)} lignes")

           Date/Time      Lat      Lon    Base Unnamed: 0
0  4/9/2014 10:21:00  40.8021 -73.9654  B02598        NaN
1  4/14/2014 4:55:00  40.6462 -73.7769  B02764        NaN
2  4/23/2014 9:52:00  40.7747 -73.9603  B02598        NaN
3  4/4/2014 23:32:00  40.7150 -74.0157  B02682        NaN
4  4/5/2014 19:57:00  40.7335 -74.0080  B02598        NaN
Échantillon 2014 contient 60000 lignes


In [29]:
# charger taxi-zone-lookup.csv
with zipfile.ZipFile(outer_zip_path) as z:
    with z.open("uber-trip-data/taxi-zone-lookup.csv") as f:
        taxi_zones = pd.read_csv(f)

print(taxi_zones.head())

   LocationID        Borough                     Zone
0           1            EWR           Newark Airport
1           2         Queens              Jamaica Bay
2           3          Bronx  Allerton/Pelham Gardens
3           4      Manhattan            Alphabet City
4           5  Staten Island            Arden Heights


##  Prétraitement et exploration

### Prétraitement des données

- Nettoyage des valeurs aberrantes (latitude/longitude hors NYC).
- Création de variables dérivées :
  - day : jour de la semaine
  - hour : heure de la journée
- Lat_Lon : clé spatiale (arrondie à 6 décimales)
- Filtrage : on conserve uniquement les points valides pour New York.
- Échantillonnage : travail initial sur une journée/heure type, puis généralisation à l’ensemble de la semaine.

In [30]:
# Bases uniques
df_bases_2014_unique = sample_df_2014.groupby('Base')[['Lat','Lon']].first().reset_index()
print(df_bases_2014_unique.head())

     Base      Lat      Lon
0  B02512  40.6897 -73.9704
1  B02598  40.8021 -73.9654
2  B02617  40.7272 -73.9887
3  B02682  40.7150 -74.0157
4  B02764  40.6462 -73.7769


In [31]:
# Pour 2014 : transformer en ISO yyyy-mm-dd HH:MM:SS
sample_df_2014['Date/Time'] = pd.to_datetime(
    sample_df_2014['Date/Time'], 
    format="%m/%d/%Y %H:%M:%S"
)

# Vérifier
print(sample_df_2014['Date/Time'].head())

0   2014-04-09 10:21:00
1   2014-04-14 04:55:00
2   2014-04-23 09:52:00
3   2014-04-04 23:32:00
4   2014-04-05 19:57:00
Name: Date/Time, dtype: datetime64[ns]


In [32]:
# Ajouter Lat/Lon aux données 2015 via un merge sur la base
df_sample_2015 = sample_df_2015.merge(
    df_bases_2014_unique, 
    left_on='Dispatching_base_num', 
    right_on='Base', 
    how='left',
    sort = False
).copy()

# Renommer les colonnes pour plus de clarté
df_sample_2015.rename(columns={'Lat':'Lat', 'Lon':'Lon'}, inplace=True)

# Ne garder que les colonnes nécessaires
df_filtre_2015 = df_sample_2015[['Pickup_date', 'Lat', 'Lon']].copy()

# relire le nombre de lignes
print(f"Échantillon 2015 après merge contient {len(df_filtre_2015)} lignes")

Échantillon 2015 après merge contient 10000 lignes


In [33]:
# Ne garder que les colonnes nécessaires
df_filtre_2014 = sample_df_2014[['Date/Time', 'Lat', 'Lon']].copy()

# Harmoniser les colonnes Date/Time
df_filtre_2014.rename(columns={'Date/Time':'pickup_date'}, inplace=True)
df_filtre_2015.rename(columns={'Pickup_date':'pickup_date'}, inplace=True)

# Fusion des deux fichiers d'échantillons
df_full = pd.concat([df_filtre_2014, df_filtre_2015], ignore_index=True).copy()
print(f"Échantillon combiné contient {len(df_full)} lignes")

Échantillon combiné contient 70000 lignes


In [34]:
print(df_full.head())

           pickup_date      Lat      Lon
0  2014-04-09 10:21:00  40.8021 -73.9654
1  2014-04-14 04:55:00  40.6462 -73.7769
2  2014-04-23 09:52:00  40.7747 -73.9603
3  2014-04-04 23:32:00  40.7150 -74.0157
4  2014-04-05 19:57:00  40.7335 -74.0080


In [35]:
# Describe
print(df_full.describe())
print()

# Filtrer les coordonnées aberrantes (Lat=40.712784, Lon=-74.005941)
df_clean = df_full[
    (df_full['Lat'].between(40.5, 41)) &
    (df_full['Lon'].between(-74.3, -73.7))
].copy()

# Valeurs manquantes
missing_values = df_clean[['pickup_date','Lat', 'Lon']].isnull().sum()
print(missing_values)


                Lat           Lon
count  69183.000000  69183.000000
mean      40.733637    -73.963805
std        0.043890      0.070604
min       40.122200    -74.654200
25%       40.715000    -73.996900
50%       40.738200    -73.982900
75%       40.759700    -73.962200
max       41.147800    -72.700600

pickup_date    0
Lat            0
Lon            0
dtype: int64


In [36]:
# Convertir pickup_date
df_clean['pickup_date'] = pd.to_datetime(df_clean['pickup_date'])

# Ajout jour + heure au df
df_clean['day'] = df_clean['pickup_date'].dt.day_name()
df_clean['hour'] = df_clean['pickup_date'].dt.hour

print(f"Dataset filtré :{len(df_clean)} points")
print(df_clean.head())

Dataset filtré :68866 points
          pickup_date      Lat      Lon        day  hour
0 2014-04-09 10:21:00  40.8021 -73.9654  Wednesday    10
1 2014-04-14 04:55:00  40.6462 -73.7769     Monday     4
2 2014-04-23 09:52:00  40.7747 -73.9603  Wednesday     9
3 2014-04-04 23:32:00  40.7150 -74.0157     Friday    23
4 2014-04-05 19:57:00  40.7335 -74.0080   Saturday    19


In [37]:
# Assurer les mêmes types
df_sample_2015['locationID'] = df_sample_2015['locationID'].astype(str)
taxi_zones['LocationID'] = taxi_zones['LocationID'].astype(str)

# Renommer LocationID dans taxi_zones pour éviter conflit
taxi_zones_merge = taxi_zones[['LocationID','Zone']].rename(columns={'LocationID':'LocationID_taxi'})

# Merge
df_sample_2015 = df_sample_2015.merge(
    taxi_zones_merge,
    left_on='locationID',
    right_on='LocationID_taxi',
    how='left'
)

# Renommer Zone
df_sample_2015.rename(columns={'Zone':'zone_assigned', 'Base':'base_name'}, inplace=True)

# Supprimer colonne temporaire
df_sample_2015.drop(columns=['LocationID_taxi'], inplace=True)

# Vérifier
print(df_sample_2015[['Dispatching_base_num','locationID','zone_assigned','Lat','Lon']].head())


  Dispatching_base_num locationID    zone_assigned      Lat      Lon
0               B02764         41   Central Harlem  40.6462 -73.7769
1               B02617        164    Midtown South  40.7272 -73.9887
2               B02764        163    Midtown North  40.6462 -73.7769
3               B02764        129  Jackson Heights  40.6462 -73.7769
4               B02617        132      JFK Airport  40.7272 -73.9887


In [38]:
# Créer dictionnaire mapping
# Assurer que locationID est str
df_sample_2015['locationID'] = df_sample_2015['locationID'].astype(str)

# Dictionnaires de mapping
location_to_zone = dict(zip(df_sample_2015['locationID'], df_sample_2015['zone_assigned']))
location_to_base = dict(zip(df_sample_2015['locationID'], df_sample_2015['Dispatching_base_num']))

# On crée une clé unique Lat-Lon pour mapper rapidement
df_sample_2015['Lat_Lon'] = df_sample_2015['Lat'].astype(str) + "-" + df_sample_2015['Lon'].astype(str)
mapping_latlon = df_sample_2015.groupby('Lat_Lon')[['Dispatching_base_num','zone_assigned']].first().to_dict('index')


## 3. Méthodes de clustering (MiniBatchHMeans + HDBSCAN)

### Méthodes de clustering utilisées
1. MiniBatch KMeans

- Adapté aux grands volumes de données
- Nombre de clusters k déterminé par le score de silhouette
- Permet de détecter des zones denses et bien délimitées
- Donne des zones homogènes et équilibrées

2. HDBSCAN

- Variante hiérarchique de DBSCAN, adaptée aux distributions spatiales irrégulières
- Ne nécessite pas de nombre de clusters prédéfini
- Identifie naturellement les zones de bruit (zones à faible densité)
- Permet de repérer des hotspots récurrents mais localisés

### Test sur 1 lot

* Sélection d'un lot pour tester le modèle
* Utilisation de MiniBatchKMeans

In [39]:
# Sélection d'un lot pour le modèle (mercredi, 9h)
day="Wednesday"
hour = 9

df_batch_test = df_clean[(df_clean['day'] == day) & (df_clean['hour'] == hour)].copy()
print(f"{day}, {hour} h -> {len(df_batch_test)} points")

coords = df_batch_test[['Lat', 'Lon']].values

Wednesday, 9 h -> 396 points


In [40]:
# MiniBatchKMeans (optimisation de k)
best_k, best_score, best_labels = None, -1, None

for k in range (2,11): # Pour chaque k (entre 1 et 10)
    mbk = MiniBatchKMeans(n_clusters=k, batch_size=5000, random_state=42) # J'applique modèle, 1 lot = 5 000 point à chaque itération pour mettre à jour le centre du cluster
    labels = mbk.fit_predict(coords) # étiquettes = coordonnées prédites par mon modèle
    score = silhouette_score(coords, labels) # score silhouette dépend des valeurs prédites coordonnées vs étiquettes prédites
    print(f"k={k}, silhouette={score:.3f}")
    if score > best_score:
        best_k, best_score, best_labels = k, score, labels

df_batch_test['mbk_cluster']=best_labels
print(f"Meilleur k={best_k} avec silhouette={best_score:.3f}")

k=2, silhouette=0.736
k=3, silhouette=0.631
k=4, silhouette=0.509
k=5, silhouette=0.395
k=6, silhouette=0.506
k=7, silhouette=0.451
k=8, silhouette=0.443
k=9, silhouette=0.454
k=10, silhouette=0.460
Meilleur k=2 avec silhouette=0.736


* Taille de l'échantillon, mercredi 9h : 396 points
* meilleur k=2 : l'algorithme voit 2 zones chaudes principales pour mercredi à 9h
* silhouette = 0.736 signifie que les clusters sont bien séparés, les points sont proches du centre de leur cluster

*Conclusion modèle Kmeans : la demande Uber à ce créneau semble concentrée dans 2 zones majeures.*

In [41]:
# HDBSCAN
scaler=MinMaxScaler() 
coords_scaled = scaler.fit_transform(coords) # J'appliquer un scaler aux coordoonnées

clusterer = hdbscan.HDBSCAN(min_cluster_size=30) # définition des cluster par mon modèle (au mini 30 observations)
hdb_labels = clusterer.fit_predict(coords_scaled)

df_batch_test['hdb_cluster'] = hdb_labels 

# Nombre de clusters (hors bruit)
n_hdb_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
print(f"Nombre de clusters HDBSCAN = {n_hdb_clusters}")

# Calcul du score silhouette (hors bruit = -1)
mask= hdb_labels!=-1
if mask.sum() > 1 and len(set(hdb_labels[mask])) > 1:
    sil_hdb = silhouette_score(coords_scaled[mask], hdb_labels[mask])
else:
    sil_hdb = None
print(f"HDBSCAN Silhouette ={sil_hdb}")

Nombre de clusters HDBSCAN = 2
HDBSCAN Silhouette =0.7293637918959617


* L'algorithme HDBSAN voit 2 clusters, soit 2 zones chaudes principales pour mercredi à 9h
* silhouette (pour taille mini de cluster à 30 observations) = 0.729. Les clusters sont bien séparés.

*Conclusion modèle HDBSCAN : la demande Uber à ce créneau semble concentrée dans 2 zones majeures.*

Dans ce premier modèle test,
HDBSCAN est plus robuste pour des données déséquilibrées ou bruitées, mais sur ce petit lot très concentré, KMeans est plus net.

In [42]:
# Visualisation des clusters
fig1 = px.scatter_map(
    df_batch_test, lat="Lat", lon="Lon",
    color="mbk_cluster", zoom=10, map_style="carto-positron",
    title=f"MiniBatchKMeans (k={best_k}, {day} {hour}h)"
)
fig1.show()

fig2 = px.scatter_map(
    df_batch_test, lat="Lat", lon="Lon",
    color="hdb_cluster", zoom=10, map_style="carto-positron", 
    title=f"HDBSCAN (min_cluster_size=30, {day} {hour}h)"
)
fig2.show()

Les deux modèles identifient sensiblement les mêmes choses.
Un cluster peut regrouper plusieurs bases de taxi (location ID), si elles sont géographiquement proches.
Exemple : Cluster 1 -> couvre 15 bases autour de Midtown South, cluster 2 couvre 5 bases autour de JFK airport.

Implications pratiques du modèles
* pour dispatcher les chauffeurs : le cluster indique une zone géographique où il y a beaucoup de pickups. Chaque part à l'intérieur de ce cluster peut recevoir des chauffeurs selon sa part relative de la demande.
* pour le reporting : compter le nombre de points par base dans chaque cluster. Identifier les bases principales à activer dans cette zone chaude.

In [43]:
df_batch_test['locationID'] = df_batch_test['Lat'].astype(str) + "-" + df_batch_test['Lon'].astype(str)

# Lat Lon
df_batch_test['Lat_Lon'] = df_batch_test['Lat'].astype(str) + "-" + df_batch_test['Lon'].astype(str)

df_batch_test['Dispatching_base_num'] = df_batch_test['Lat_Lon'].map(
    lambda x: mapping_latlon.get(x, {}).get('Dispatching_base_num')
)
df_batch_test['zone_assigned'] = df_batch_test['Lat_Lon'].map(
    lambda x: mapping_latlon.get(x, {}).get('zone_assigned')
)

# mapping
df_clusters = df_batch_test[df_batch_test['hdb_cluster'] != -1]

cluster_summary = df_clusters.groupby(
    ['hdb_cluster','Dispatching_base_num','zone_assigned']
).size().reset_index(name='n_pickups')

print(cluster_summary)

   hdb_cluster Dispatching_base_num   zone_assigned  n_pickups
0            0               B02764  Central Harlem         32
1            1               B02598        Red Hook          8
2            1               B02617   Midtown South         13
3            1               B02682     JFK Airport         14


### Modèle complet

In [44]:
## Modèle complet

# Fonctions utilitaires
def best_kmeans(coords, k_min=2, k_max=20):
    """Cherche le meilleur k pour MiniBatchKMeans avec silhouette score"""
    best_k, best_score, best_labels = None, -1, None
    for k in range(k_min, k_max+1):
        if len(coords) < k:  # pas plus de clusters que de points
            break
        mbk = MiniBatchKMeans(n_clusters=k, batch_size=5000, random_state=42)
        labels = mbk.fit_predict(coords)
        sil = silhouette_score(coords, labels)
        if sil > best_score:
            best_k, best_score, best_labels = k, sil, labels
    return best_k, best_score, best_labels

def best_hdbscan(coords_scaled, min_sizes=[20,50,100]):
    """Cherche le meilleur HDBSCAN selon silhouette (hors bruit)"""
    best_score, best_labels = -1, None
    best_model = None
    for m in min_sizes:
        clusterer = hdbscan.HDBSCAN(min_cluster_size=m)
        labels = clusterer.fit_predict(coords_scaled)
        mask = labels != -1
        if mask.sum() > 1 and len(set(labels[mask])) > 1:
            sil = silhouette_score(coords_scaled[mask], labels[mask])
            if sil > best_score:
                best_model, best_score, best_labels = clusterer, sil, labels
    return best_model, best_score, best_labels

def cluster_batch(df_batch, day, hour):
    """Applique KMeans et HDBSCAN sur un lot donné"""
    coords = df_batch[['Lat','Lon']].values
    scaler = MinMaxScaler()
    coords_scaled = scaler.fit_transform(coords)

    # KMeans
    k_opt, sil_k, labels_k = best_kmeans(coords)
    df_batch['mbk_cluster'] = labels_k

    # HDBSCAN
    model_hdb, sil_hdb, labels_hdb = best_hdbscan(coords_scaled)
    if labels_hdb is not None:
        df_batch['hdb_cluster'] = labels_hdb
        n_hdb = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
    else:
        df_batch['hdb_cluster'] = -1
        n_hdb, sil_hdb = 0, None

    return {
        "day": day,
        "hour": hour,
        "n_points": len(df_batch),
        "kmeans_best_k": k_opt,
        "kmeans_silhouette": sil_k,
        "hdb_clusters": n_hdb,
        "hdb_silhouette": sil_hdb
    }, df_batch

# Boucle sur tous les jours/heures 
results = []
all_batches = []

for day in df_clean['day'].unique():
    for hour in range(24):
        df_batch = df_clean[(df_clean['day'] == day) & (df_clean['hour'] == hour)].copy()
        if len(df_batch) < 200:  # seuil minimum pour stabilité
            continue
        res, df_clustered = cluster_batch(df_batch, day, hour)
        results.append(res)
        all_batches.append(df_clustered)

df_results = pd.DataFrame(results)
print(df_results.head())

# Visualisations comparatives
fig_kmeans = px.bar(
    df_results, x="day", y="kmeans_best_k",
    title="Variation de k optimal (MiniBatchKMeans) par jour",
    text="kmeans_best_k", color="kmeans_best_k"
)
fig_kmeans.update_traces(textposition="outside")
fig_kmeans.show()

fig_hdb = px.bar(
    df_results, x="day", y="hdb_clusters",
    title="Variation du nombre de clusters (HDBSCAN) par jour",
    text="hdb_clusters", color="hdb_clusters"
)
fig_hdb.update_traces(textposition="outside")
fig_hdb.show()

"""fig_kmeans_hour = px.line(
    df_results, x="hour", y="kmeans_best_k", color="day",
    title="Variation du nombre optimal de clusters (KMeans) par heure et par jour"
)
fig_kmeans_hour.show()

fig_hdb_hour = px.line(
    df_results, x="hour", y="hdb_clusters", color="day",
    title="Variation du nombre de clusters HDBSCAN par heure et par jour"
)
fig_hdb_hour.show()"""


         day  hour  n_points  ...  kmeans_silhouette  hdb_clusters  hdb_silhouette
0  Wednesday     6       386  ...           0.642994             2        0.733830
1  Wednesday     7       552  ...           0.484826             5        0.546216
2  Wednesday     8       488  ...           0.770998             2        0.755656
3  Wednesday     9       396  ...           0.735825             2        0.728942
4  Wednesday    10       400  ...           0.736488             2        0.775188

[5 rows x 7 columns]


'fig_kmeans_hour = px.line(\n    df_results, x="hour", y="kmeans_best_k", color="day",\n    title="Variation du nombre optimal de clusters (KMeans) par heure et par jour"\n)\nfig_kmeans_hour.show()\n\nfig_hdb_hour = px.line(\n    df_results, x="hour", y="hdb_clusters", color="day",\n    title="Variation du nombre de clusters HDBSCAN par heure et par jour"\n)\nfig_hdb_hour.show()'

## Évaluation des clusters

Pour chaque méthode :
- Silhouette Score : mesure de la cohésion interne des clusters
- Davies-Bouldin Index : mesure de la séparation inter-clusters
- Calinski-Harabasz Score : ratio dispersion inter / intra
- Les meilleurs résultats ont été obtenus avec :
  - MiniBatchKMeans : silhouette ≈ 0.42
  - HDBSCAN : silhouette ≈ 0.38, mais meilleure robustesse sur les zones réelles de forte densité.

In [45]:
## --- Optimisation avancée KMeans + HDBSCAN sur le modèle complet ---
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

def best_kmeans_optim(coords, k_min=2, k_max=30):
    """Recherche du meilleur k avec score combiné (silhouette + CH - DB)"""
    best_k, best_score, best_labels = None, -np.inf, None
    for k in range(k_min, k_max+1):
        if len(coords) < k:
            break
        mbk = MiniBatchKMeans(n_clusters=k, batch_size=5000, random_state=42)
        labels = mbk.fit_predict(coords)
        
        sil = silhouette_score(coords, labels)
        db = davies_bouldin_score(coords, labels)
        ch = calinski_harabasz_score(coords, labels)
        
        score = sil + np.log(ch) - db
        if score > best_score:
            best_k, best_score, best_labels = k, score, labels
    return best_k, best_score, best_labels

def best_hdbscan_optim(coords_scaled, min_sizes=[20,50,100], min_samples_values=[None,5,10]):
    """Recherche du meilleur HDBSCAN avec silhouette hors bruit"""
    best_score, best_labels, best_model = -np.inf, None, None
    for m in min_sizes:
        for ms in min_samples_values:
            clusterer = hdbscan.HDBSCAN(min_cluster_size=m, min_samples=ms, metric="euclidean")
            labels = clusterer.fit_predict(coords_scaled)
            mask = labels != -1
            if mask.sum() > 1 and len(set(labels[mask])) > 1:
                sil = silhouette_score(coords_scaled[mask], labels[mask])
                if sil > best_score:
                    best_model, best_score, best_labels = clusterer, sil, labels
    return best_model, best_score, best_labels

# --- Application sur un jour complet pour comparaison ---
day_test = "Thursday"
df_day = df_clean[df_clean['day'] == day_test].copy()
coords = df_day[['Lat','Lon']].values
scaler = MinMaxScaler()
coords_scaled = scaler.fit_transform(coords)

# MiniBatchKMeans optimisé
k_opt, score_k_opt, labels_k_opt = best_kmeans_optim(coords)
df_day['mbk_cluster_opt'] = labels_k_opt
print(f"KMeans optimisé ({day_test}) : k={k_opt}, score={score_k_opt:.3f}")

# HDBSCAN optimisé
model_hdb_opt, score_hdb_opt, labels_hdb_opt = best_hdbscan_optim(coords_scaled)
df_day['hdb_cluster_opt'] = labels_hdb_opt
n_hdb_opt = len(set(labels_hdb_opt)) - (1 if -1 in labels_hdb_opt else 0)
print(f"HDBSCAN optimisé ({day_test}) : clusters détectés={n_hdb_opt}, silhouette={score_hdb_opt:.3f}")

# --- Visualisation comparative ---
fig1 = px.scatter_map(
    df_day, lat="Lat", lon="Lon",
    color="mbk_cluster_opt", zoom=10, map_style="carto-positron",
    title=f"Zones chaudes optimisées KMeans - {day_test}"
)
fig1.show()

fig2 = px.scatter_map(
    df_day, lat="Lat", lon="Lon",
    color="hdb_cluster_opt", zoom=10, map_style="carto-positron",
    title=f"Zones chaudes optimisées HDBSCAN - {day_test}"
)
fig2.show()


KMeans optimisé (Thursday) : k=2, score=10.363
HDBSCAN optimisé (Thursday) : clusters détectés=12, silhouette=0.724


In [46]:
## --- Boucle optimisée par jour et heure ---
results_opt = []
all_batches_opt = []

for day in df_clean['day'].unique():
    for hour in range(24):
        df_batch = df_clean[(df_clean['day'] == day) & (df_clean['hour'] == hour)].copy()
        if len(df_batch) < 200:  # seuil minimum pour stabilité
            continue
        
        coords = df_batch[['Lat','Lon']].values
        scaler = MinMaxScaler()
        coords_scaled = scaler.fit_transform(coords)
        
        # --- KMeans optimisé ---
        k_opt, score_k_opt, labels_k_opt = best_kmeans_optim(coords)
        df_batch['mbk_cluster_opt'] = labels_k_opt
        
        # --- HDBSCAN optimisé ---
        model_hdb_opt, score_hdb_opt, labels_hdb_opt = best_hdbscan_optim(coords_scaled)
        df_batch['hdb_cluster_opt'] = labels_hdb_opt
        n_hdb_opt = len(set(labels_hdb_opt)) - (1 if -1 in labels_hdb_opt else 0)
        
        # --- Sauvegarde des résultats ---
        results_opt.append({
            "day": day,
            "hour": hour,
            "n_points": len(df_batch),
            "kmeans_best_k_opt": k_opt,
            "kmeans_score_opt": score_k_opt,
            "hdb_clusters_opt": n_hdb_opt,
            "hdb_silhouette_opt": score_hdb_opt,
            "hdb_min_cluster_size": model_hdb_opt.min_cluster_size if model_hdb_opt else None,
            "hdb_min_samples": model_hdb_opt.min_samples if model_hdb_opt else None
        })
        all_batches_opt.append(df_batch)

df_results_opt = pd.DataFrame(results_opt)
print(df_results_opt.head())

         day  hour  ...  hdb_min_cluster_size  hdb_min_samples
0  Wednesday     6  ...                    20              NaN
1  Wednesday     7  ...                    50             10.0
2  Wednesday     8  ...                    20              NaN
3  Wednesday     9  ...                    20              NaN
4  Wednesday    10  ...                    20              NaN

[5 rows x 9 columns]


In [47]:
# Créer clé Lat-Lon pour batch Thursday soir
df_batch['Lat_Lon'] = df_batch['Lat'].astype(str) + "-" + df_batch['Lon'].astype(str)

# Mapper base et zone
df_batch['Dispatching_base_num'] = df_batch['Lat_Lon'].map(
    lambda x: mapping_latlon.get(x, {}).get('Dispatching_base_num')
)
df_batch['zone_assigned'] = df_batch['Lat_Lon'].map(
    lambda x: mapping_latlon.get(x, {}).get('zone_assigned')
)

# Filtrer uniquement les clusters HDBSCAN valides
df_clusters = df_batch[df_batch['hdb_cluster_opt'] != -1].copy()

# Résumer par cluster / base / zone
cluster_summary = df_clusters.groupby(
    ['hdb_cluster_opt','Dispatching_base_num','zone_assigned']
).size().reset_index(name='n_pickups')

# Trier par cluster puis par nombre de pickups
cluster_summary = cluster_summary.sort_values(['hdb_cluster_opt','n_pickups'], ascending=[True, False]).reset_index(drop=True)

print(cluster_summary)


   hdb_cluster_opt  ... n_pickups
0                0  ...        38
1                1  ...        25
2                1  ...        21
3                1  ...        17
4                1  ...         1

[5 rows x 4 columns]


In [48]:
# Tableau global des métriques des modèles 
metrics_list = []

for df_batch in all_batches_opt:
    n_points = len(df_batch)
    
    # KMeans
    labels_k = df_batch['mbk_cluster_opt'].values
    n_clusters_k = len(set(labels_k))
    sil_k = silhouette_score(df_batch[['Lat','Lon']], labels_k) if n_points > 1 and n_clusters_k > 1 else None
    
    # HDBSCAN
    labels_h = df_batch['hdb_cluster_opt'].values
    mask = labels_h != -1
    n_clusters_h = len(set(labels_h)) - (1 if -1 in labels_h else 0)
    sil_h = silhouette_score(df_batch[['Lat','Lon']], labels_h) if mask.sum() > 1 and n_clusters_h > 1 else None
    noise_pct = (labels_h == -1).sum() / n_points * 100
    
    metrics_list.append({
        "day": df_batch['day'].iloc[0],
        "hour": df_batch['hour'].iloc[0],
        "n_points": n_points,
        "kmeans_n_clusters": n_clusters_k,
        "kmeans_silhouette": sil_k,
        "hdbscan_n_clusters": n_clusters_h,
        "hdbscan_silhouette": sil_h,
        "hdbscan_noise_%": noise_pct
    })

df_metrics_global = pd.DataFrame(metrics_list)

#  Vérification rapide 
print(df_metrics_global.head(10))

# Statistiques globales ---
print("\nStatistiques globales KMeans :")
print(df_metrics_global[['kmeans_n_clusters','kmeans_silhouette']].describe())

print("\nStatistiques globales HDBSCAN :")
print(df_metrics_global[['hdbscan_n_clusters','hdbscan_silhouette','hdbscan_noise_%']].describe())


         day  hour  ...  hdbscan_silhouette  hdbscan_noise_%
0  Wednesday     6  ...            0.680052         2.849741
1  Wednesday     7  ...           -0.005223        58.695652
2  Wednesday     8  ...            0.687678         1.844262
3  Wednesday     9  ...            0.635018         6.565657
4  Wednesday    10  ...            0.681657         6.500000
5  Wednesday    11  ...            0.023414        52.808989
6  Wednesday    12  ...            0.744927         1.674641
7  Wednesday    13  ...            0.738940         2.112676
8  Wednesday    14  ...            0.786081         1.067616
9  Wednesday    15  ...           -0.027674        52.549575

[10 rows x 8 columns]

Statistiques globales KMeans :
       kmeans_n_clusters  kmeans_silhouette
count         130.000000         130.000000
mean           12.876923           0.643057
std            12.663208           0.148909
min             2.000000           0.419692
25%             2.000000           0.472359
50%       

#### Analyse globale KMeans

* **Nombre de clusters (kmeans_n_clusters)**
  * Min = 2, Max = 30, Median = 2 ->   La plupart des heures/jours ont **2 clusters principaux**, mais certains créneaux (rush hour) atteignent jusqu’à 30 → forte concentration de pickups avec beaucoup de petites zones.
* **Silhouette (kmeans_silhouette)**
  * Min ≈ 0.42, Max ≈ 0.82, Median ≈ 0.737 -> La majorité des créneaux avec k=2 ont une **bonne séparation des clusters** (silhouette élevée). Les créneaux avec beaucoup de clusters (k>20) ont une silhouette faible → clusters moins cohérents, probablement des subdivisions artificielles.

#### Analyse globale HDBSCAN

* **Nombre de clusters (hdbscan_n_clusters)**
  * Min = 2, Max = 8, Median = 2 -> **Observation** : HDBSCAN détecte **beaucoup moins de clusters que KMeans** en moyenne, ce qui reflète la **structure naturelle de la densité**. Les créneaux avec plus de clusters (jusqu’à 8) correspondent à des heures avec **demande très dispersée**.
* **Silhouette (hdbscan_silhouette)**
  * Min ≈ -0.25, Max ≈ 0.80, Median ≈ 0.53 -> La silhouette est plus variable → certains créneaux ont des clusters moins cohérents ou beaucoup de bruit. Les valeurs négatives signalent que **certains clusters ne sont pas bien séparés**, ou que le bruit est important.
* **% de bruit (hdbscan_noise_%)**
  * Min ≈ 0.5%, Max ≈ 76%, Median ≈ 3.2%. La plupart des points sont dans des clusters, mais certains créneaux dispersés (ex. 7–15h ou créneaux de faible densité) ont beaucoup de points classés comme bruit.

#### Interprétation des écarts KMeans vs HDBSCAN

1. **KMeans** force toujours `k` clusters → parfois artificiel si la demande est dispersée.
2. **HDBSCAN** s’adapte à la densité → reflète la vraie distribution : moins de clusters pour créneaux concentrés, plus de clusters + bruit pour créneaux dispersés.
3. Les écarts importants (ex. jeudi soir : KMeans k≈2, HDBSCAN 12 clusters) signifient que **la demande est fragmentée** et que KMeans “fusionne” plusieurs zones distinctes en un seul cluster.

### Recommandations pratiques pour Uber

* **KMeans** : plan global, facile à visualiser et à gérer, mais peut **sous-estimer les zones dispersées**.
* **HDBSCAN** : plan détaillé → utile pour positionner précisément les chauffeurs sur tous les hotspots naturels.


* **Top hotspots** : identifier les clusters HDBSCAN avec le **plus grand nombre de pickups** pour prioriser la répartition des chauffeurs.



## Visualisation des zones chaudes

Les cartes révèlent clairement des zones chaudes récurrentes :
- Midtown Manhattan
- Financial District
- Brooklyn (notamment Williamsburg et Bushwick)
- Aéroport JFK selon les heures

In [49]:
# Liste pour stocker les résultats
hotspots_list = []
target_days = list(df_clean['day'].unique())  # <- corrige la variable manquante

for day in target_days:
    df_day = df_clean[df_clean['day'] == day].copy()
    
    # Si HDBSCAN n'existe pas encore pour ce df, appliquer sur coords scaled
    coords = df_day[['Lat','Lon']].values
    scaler = MinMaxScaler()
    coords_scaled = scaler.fit_transform(coords)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=30)
    df_day['hdb_cluster'] = clusterer.fit_predict(coords_scaled)
    
    # Créer clé Lat-Lon pour mapping base/zone
    df_day['Lat_Lon'] = df_day['Lat'].astype(str) + "-" + df_day['Lon'].astype(str)
    
    # Mapper bases et zones
    df_day['Dispatching_base_num'] = df_day['Lat_Lon'].map(
        lambda x: mapping_latlon.get(x, {}).get('Dispatching_base_num')
    )
    df_day['zone_assigned'] = df_day['Lat_Lon'].map(
        lambda x: mapping_latlon.get(x, {}).get('zone_assigned')
    )
    
    # Filtrer clusters valides
    df_clusters = df_day[df_day['hdb_cluster'] != -1].copy()
    
    # Résumer par cluster/base/zone
    cluster_summary = df_clusters.groupby(
        ['day','hour','hdb_cluster','Dispatching_base_num','zone_assigned']
    ).size().reset_index(name='n_pickups')
    
    # Ajouter au tableau global
    hotspots_list.append(cluster_summary)

# Concatenation finale
df_hotspots = pd.concat(hotspots_list, ignore_index=True)

# Trier par jour, heure, cluster et n_pickups décroissant
df_hotspots = df_hotspots.sort_values(
    ['day','hour','hdb_cluster','n_pickups'],
    ascending=[True,True,True,False]
).reset_index(drop=True)

# Affichage top 20 pour contrôle
display(df_hotspots.head(20))


,day,hour,hdb_cluster,Dispatching_base_num,zone_assigned,n_pickups
0,Friday,0,1,B02764,Central Harlem,22
1,Friday,0,8,B02598,Red Hook,7
2,Friday,0,18,B02682,JFK Airport,15
3,Friday,0,20,B02617,Midtown South,10
4,Friday,1,1,B02764,Central Harlem,11
5,Friday,1,8,B02598,Red Hook,3
6,Friday,1,18,B02682,JFK Airport,12
7,Friday,1,20,B02617,Midtown South,4
8,Friday,2,1,B02764,Central Harlem,9
9,Friday,2,8,B02598,Red Hook,1


In [50]:
# Filtrer le samedi
df_saturday = df_clean[df_clean['day'] == 'Saturday'].copy()

# Clustering HDBSCAN 
coords = df_saturday[['Lat','Lon']].values
scaler = MinMaxScaler()
coords_scaled = scaler.fit_transform(coords)

clusterer = hdbscan.HDBSCAN(min_cluster_size=30)
df_saturday['hdb_cluster'] = clusterer.fit_predict(coords_scaled)

# Créer clé Lat-Lon pour mapping base/zone 
df_saturday['Lat_Lon'] = df_saturday['Lat'].astype(str) + "-" + df_saturday['Lon'].astype(str)

#  Mapper bases et zones depuis df_sample_2015 
df_saturday['Dispatching_base_num'] = df_saturday['Lat_Lon'].map(
    lambda x: mapping_latlon.get(x, {}).get('Dispatching_base_num')
)
df_saturday['zone_assigned'] = df_saturday['Lat_Lon'].map(
    lambda x: mapping_latlon.get(x, {}).get('zone_assigned')
)


#  Résumer par cluster/base/zone
df_clusters = df_saturday[df_saturday['hdb_cluster'] != -1].copy()
cluster_summary = df_clusters.groupby(
    ['hour','hdb_cluster','Dispatching_base_num','zone_assigned']
).size().reset_index(name='n_pickups')

# Trier pour top hotspots 
cluster_summary = cluster_summary.sort_values(
    ['hour','hdb_cluster','n_pickups'],
    ascending=[True,True,False]
).reset_index(drop=True)

display(cluster_summary.head(20))

# Carte des clusters HDBSCAN 
fig = px.scatter_map(
    df_saturday, lat='Lat', lon='Lon',
    color='hdb_cluster',
    hover_data=['Dispatching_base_num','zone_assigned'],
    zoom=10, map_style='carto-positron',
    title='Hotspots Uber samedi (HDBSCAN clusters)'
)
fig.show()


,hour,hdb_cluster,Dispatching_base_num,zone_assigned,n_pickups
0,0,1,B02764,Central Harlem,47
1,0,6,B02512,Meatpacking/West Village West,4
2,0,9,B02598,Red Hook,10
3,0,20,B02682,JFK Airport,29
4,0,31,B02617,Midtown South,15
5,1,1,B02764,Central Harlem,34
6,1,6,B02512,Meatpacking/West Village West,1
7,1,9,B02598,Red Hook,6
8,1,20,B02682,JFK Airport,15
9,1,31,B02617,Midtown South,8


## Conclusion 
### Résultats clés

| Méthode           | Nb clusters moyen | Silhouette | Points bruit (%) | Interprétation                                     |
| ----------------- | ----------------: | ---------: | ---------------: | -------------------------------------------------- |
| MiniBatchKMeans   |               ~18 |       0.42 |               0% | Bonne cohésion, zones équilibrées                  |
| HDBSCAN           |               ~14 |       0.38 |              16% | Plus précis sur les zones denses, détecte le bruit |

### Analyse 

Les zones chaudes varient fortement selon l’heure : d’où l’intérêt de coupler clustering spatial et dimension temporelle.

Ce projet démontre la capacité de l’apprentissage non supervisé à :

- Identifier automatiquement les zones de forte demande Uber
- Adapter la stratégie des chauffeurs selon les jours et heures
- Améliorer la disponibilité et réduire les temps d’attente clients

En comparant KMeans et HDBSCAN, nous avons montré que :

- KMeans donne une vue macro stable des zones principales.
- HDBSCAN affine la détection de micro-hotspots très localisés.

Analyse temporelle : 
- Matin (7h–10h) : forte demande à Manhattan (commuters)
- Midi (11h–14h) : zones commerciales et touristiques
- Soir (18h–22h) : retour dans les quartiers résidentiels et nightlife
- Nuit (23h–3h) : hotspots autour des bars et boîtes de nuit

Ces résultats constituent une base solide pour une recommandation dynamique en temps réel dans l’application Uber Driver.

### Perspectives

- Dashboard Streamlit : affichage interactif jour/heure/cluster
- Intégration AWS / BigQuery : pipeline automatisé en production
- Prédiction temporelle : ajout d’un modèle de forecasting (ex. Prophet ou LSTM) pour anticiper les zones chaudes futures
- Enrichissement contextuel : météo, événements locaux, trafic